
# 03 — Causal Data Preparation

## Objective
Prepare the Online Retail II transaction data for causal analysis of
the relationship between product unit price and quantity purchased.

## Research question
For comparable transactions of the same product, how does a change
in observed unit price relate to the quantity purchased?

## Variable definitions
- Treatment: `unit_price`
- Outcome: `quantity`
- Product identifier: `stockcode`
- Time variable: `invoicedate`
- Geographic variable: `country`
- Customer identifier: `customer_id` (partially missing)

## Important limitation
This is observational transaction data. Inventory availability,
unrecorded promotions, and other demand-related factors may influence
both price and quantity. Double Machine Learning does not automatically
remove bias from unmeasured confounding.


In [6]:

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Data" / "Processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "online_retail_sales_initial.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "Data" / "Processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Input dataset:", DATA_PATH)


Project root: c:\intern_project\econocausal-dml
Input dataset: c:\intern_project\econocausal-dml\Data\Processed\online_retail_sales_initial.csv


In [7]:

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["invoicedate"],
    dtype={"invoice": "string", "stockcode": "string"}
)

required_columns = [
    "invoice", "stockcode", "quantity", "invoicedate",
    "unit_price", "customer_id", "country", "line_sales_value"
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Shape:", df.shape)
print("Date range:", df["invoicedate"].min(), "to", df["invoicedate"].max())
display(df.head())


Shape: (1041670, 9)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


,invoice,stockcode,description,quantity,invoicedate,unit_price,customer_id,country,line_sales_value
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0



## Treatment and outcome

The observed unit price is the treatment variable, and the quantity
purchased on each invoice line is the outcome.

Price is continuous rather than binary. For the initial model, we will
estimate how the expected quantity changes with price, conditional on
the measured covariates.

A causal interpretation requires adequate overlap, correct modelling
assumptions, and no important unmeasured confounding.


In [8]:

quality_checks = pd.Series({
    "rows": len(df),
    "unique_products": df["stockcode"].nunique(),
    "unique_invoices": df["invoice"].nunique(),
    "missing_customer_id": int(df["customer_id"].isna().sum()),
    "missing_dates": int(df["invoicedate"].isna().sum()),
    "nonpositive_quantity": int((df["quantity"] <= 0).sum()),
    "nonpositive_price": int((df["unit_price"] <= 0).sum()),
    "duplicate_rows": int(df.duplicated().sum()),
})

display(quality_checks.to_frame("value"))

assert df["invoicedate"].notna().all()
assert (df["quantity"] > 0).all()
assert (df["unit_price"] > 0).all()


,value
rows,1041670
unique_products,4917
unique_invoices,40077
missing_customer_id,236121
missing_dates,0
nonpositive_quantity,0
nonpositive_price,0
duplicate_rows,33757


In [9]:

df["year"] = df["invoicedate"].dt.year
df["month"] = df["invoicedate"].dt.month
df["year_month"] = df["invoicedate"].dt.to_period("M").astype(str)
df["day_of_week"] = df["invoicedate"].dt.dayofweek
df["hour"] = df["invoicedate"].dt.hour

display(
    df[[
        "invoicedate", "year", "month",
        "year_month", "day_of_week", "hour"
    ]].head()
)


,invoicedate,year,month,year_month,day_of_week,hour
0,2009-12-01 07:45:00,2009,12,2009-12,1,7
1,2009-12-01 07:45:00,2009,12,2009-12,1,7
2,2009-12-01 07:45:00,2009,12,2009-12,1,7
3,2009-12-01 07:45:00,2009,12,2009-12,1,7
4,2009-12-01 07:45:00,2009,12,2009-12,1,7



## Analysis sample

The full dataset is large, and price variation between different
products can reflect product quality or product type rather than a
price effect.

We will first identify products with sufficient transaction history
and observed price variation. This is a screening step, not proof of
causal identification.

We will also check whether product prices vary over time. Products
with multiple prices across different periods provide a more useful
starting point for investigating within-product price effects.


In [10]:

product_summary = (
    df.groupby("stockcode")
      .agg(
          transaction_lines=("quantity", "size"),
          unique_prices=("unit_price", "nunique"),
          unique_months=("year_month", "nunique"),
          median_price=("unit_price", "median"),
          median_quantity=("quantity", "median"),
      )
      .reset_index()
)

eligible_products = product_summary.loc[
    (product_summary["transaction_lines"] >= 30)
    & (product_summary["unique_prices"] >= 2)
    & (product_summary["unique_months"] >= 3),
    "stockcode"
]

print("All products:", df["stockcode"].nunique())
print("Products passing screening:", len(eligible_products))

display(
    product_summary[
        product_summary["stockcode"].isin(eligible_products)
    ].sort_values("transaction_lines", ascending=False).head(20)
)


All products: 4917
Products passing screening: 3459


,stockcode,transaction_lines,unique_prices,unique_months,median_price,median_quantity
4301,85123A,5686,14,25,2.95,6.0
4273,85099B,4124,14,25,1.95,10.0
1638,22423,4061,13,22,12.75,2.0
615,21212,3290,11,25,0.55,24.0
258,20725,3179,12,25,1.65,10.0
3991,84879,2938,10,25,1.69,16.0
3148,47566,2740,17,25,4.95,4.0
632,21232,2555,7,25,1.25,12.0
1604,22383,2507,9,24,1.65,10.0
260,20727,2484,9,25,1.65,6.0


In [11]:

model_df = df[df["stockcode"].isin(eligible_products)].copy()

# Keep the primary variables complete.
# Customer ID is not required for this first version.
model_df = model_df.dropna(
    subset=["unit_price", "quantity", "stockcode", "invoicedate", "country"]
)

# Log transforms may help reduce skew, while preserving positive values.
model_df["log_price"] = np.log(model_df["unit_price"])
model_df["log_quantity"] = np.log1p(model_df["quantity"])

print("Model dataset shape:", model_df.shape)
print("Products:", model_df["stockcode"].nunique())
print("Price range:", model_df["unit_price"].min(), "to", model_df["unit_price"].max())

display(model_df.head())


Model dataset shape: (1019488, 16)
Products: 3459
Price range: 0.001 to 25111.09


,invoice,stockcode,description,quantity,invoicedate,unit_price,customer_id,country,line_sales_value,year,month,year_month,day_of_week,hour,log_price,log_quantity
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,2009-12,1,7,1.938742,2.564949
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,2009-12,1,7,1.909543,2.564949
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,2009-12,1,7,1.909543,2.564949
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,2009-12,1,7,0.741937,3.891820
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,2009-12,1,7,0.223144,3.218876


In [12]:

model_path = OUTPUT_DIR / "causal_model_dataset.csv"
summary_path = OUTPUT_DIR / "causal_product_screening.csv"

model_df.to_csv(model_path, index=False)
product_summary.to_csv(summary_path, index=False)

print("Saved model dataset:", model_path)
print("Saved product screening:", summary_path)


Saved model dataset: c:\intern_project\econocausal-dml\Data\Processed\causal_model_dataset.csv
Saved product screening: c:\intern_project\econocausal-dml\Data\Processed\causal_product_screening.csv



## Interpretation and limitations

The screening criteria identify products with repeated transactions,
price variation, and observations across multiple months.

However:
- Observed price changes may be associated with promotions, product
  changes, customer composition, or other demand factors.
- Inventory availability is not measured.
- Transaction lines are not necessarily independent observations.
- Large quantities and unusual prices should be investigated rather
  than automatically deleted.
- A causal estimate should be reported with sensitivity and robustness
  checks, not as definitive proof of price elasticity.

Next: construct the causal graph, define the estimand, and assess
overlap before fitting a Double Machine Learning model.
